# Train Model B on EuroSAT with a Colab GPU
**Contribution: KavijaK.** This notebook calls the repository scripts; it does not duplicate the model or redefine the team's split. Teammate code/results are read without modification.

Choose **Runtime → Change runtime type → GPU**, then run cells in order. You need this notebook and the delivered **EN3150_A03_ModelB_Code.zip**. The ZIP contains the committed feature branch and shared dependencies, so GitHub login is unnecessary. Checkpoints/results persist in your Google Drive; raw images stay on fast Colab disk.

The code has local offline tests. **This notebook has not yet been executed on a Colab GPU, and no Model B EuroSAT scores are prefilled.**


In [ ]:
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime, reconnect, then rerun this cell.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
PROJECT_ROOT = Path("/content/EN3150_A03")


## 1. Upload the code archive
Upload **EN3150_A03_ModelB_Code.zip** when prompted. Keep the same archive for resuming a run. To switch source versions, start a fresh runtime and a new run ID.


In [ ]:
import io
import zipfile
from pathlib import PurePosixPath
from google.colab import files

if not (PROJECT_ROOT / "scripts/run_model_b_experiments.py").is_file():
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith(".zip")]
    if len(candidates) != 1:
        raise ValueError("Upload exactly one code ZIP.")
    with zipfile.ZipFile(io.BytesIO(uploaded[candidates[0]])) as archive:
        for info in archive.infolist():
            relative = PurePosixPath(info.filename)
            if (relative.is_absolute() or ".." in relative.parts or
                "\\" in info.filename or not relative.parts or
                relative.parts[0] != "EN3150_A03"):
                raise ValueError(f"Unexpected archive path: {info.filename}")
        archive.extractall("/content")
    del uploaded
assert (PROJECT_ROOT / "scripts/run_model_b_experiments.py").is_file()
print("Code is ready at", PROJECT_ROOT)


In [ ]:
import subprocess
import sys

def execute(arguments):
    subprocess.run([str(a) for a in arguments], cwd=PROJECT_ROOT, check=True)

# Keep Colab's installed CUDA-enabled torch/torchvision when requirements are met.
execute([sys.executable, "-m", "pip", "install", "-e", ".", "pytest"])
execute([sys.executable, "-m", "pytest", "tests/test_model_b_params.py",
         "tests/test_model_b_training.py", "tests/test_model_b_integration.py", "-q"])


## 2. Mount Drive and set a run ID
Accept the normal Drive authorization dialog. Set `RUN_ID` once and reuse it after interruptions. A new experiment needs a new ID. This folder contains **Model B outputs only**. Keep epoch count, seed, batch size and optimizer list unchanged when resuming.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RUN_ID = "modelB_gpu_01"
EPOCHS = 20
BATCH_SIZE = 64
SEED = 42
RESULTS_ROOT = Path("/content/drive/MyDrive/EN3150_A03/model_b")
RUN_DIR = RESULTS_ROOT / RUN_ID
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent results:", RUN_DIR)


## 3. Download images and verify the frozen split
Download the official RGB release from Zenodo, verify the archive checksum, and verify every image against the committed 70/15/15 split. This setup command **does not reconstruct or overwrite** split manifests and supports a different Colab PyTorch version.

Images stay under `/content/EN3150_A03/data/raw/EuroSAT_RGB`. After a runtime reset, rerun this cell to download them again. If extraction was interrupted, delete only the incomplete local image directory and rerun this setup cell.


In [ ]:
execute([sys.executable, "scripts/prepare_model_b_data.py"])


## 4. Run Adam, SGD and momentum SGD (20 epochs each)
The primary recipe matches Model A: RGB 64×64, `ToTensor()`, no augmentation or normalization, cross-entropy, batch size 64, seed 42, no scheduler/weight decay. Each optimizer resets initialization and shuffle seeds. Adam uses 0.001; SGD uses 0.01, with momentum 0.9 in the momentum run.

The minimum-validation-loss checkpoint is selected; all training finishes before test evaluation. Checkpoints and history are saved every completed epoch. When rerunning this cell, `--resume` recovers the last completed epoch and avoids repeating saved test evaluations. Resume with the same GPU model, package versions, settings, code and platform. Use a new run ID if these change.


In [ ]:
command = [sys.executable, "scripts/run_model_b_experiments.py",
           "--run-id", RUN_ID, "--results-dir", RESULTS_ROOT,
           "--epochs", EPOCHS, "--batch-size", BATCH_SIZE,
           "--seed", SEED, "--num-workers", 0, "--device", "cuda"]
if (RUN_DIR / "experiment_config.json").is_file():
    command.append("--resume")
execute(command)


## 5. Review results and read-only teammate comparison
The generated report includes optimizer curves, class metrics, confusion matrices, parameter/MAC counts and timing definitions. The comparison command reads committed Model A/MobileNetV2 metrics and writes a separate table inside this Model B run.

Do not treat epoch times from different computers as speedups. MobileNet uses pretrained weights and different preprocessing. EfficientNet-B0 is still pending in the source checkout; its results are not invented or included.


In [ ]:
import json
from IPython.display import Markdown, display

comparison_dir = RUN_DIR / "integration_comparison"
if not comparison_dir.exists():
    execute([sys.executable, "scripts/compare_model_b.py", "--model-b-run", RUN_DIR,
             "--output-dir", comparison_dir])
with (RUN_DIR / "comparison.json").open() as stream:
    summary = json.load(stream)
for row in summary:
    print(row["optimizer"], "test accuracy:", f"{row['accuracy']:.4f}",
          "macro precision:", f"{row['macro_precision']:.4f}",
          "macro recall:", f"{row['macro_recall']:.4f}")
display(Markdown((comparison_dir / "comparison.md").read_text()))
for row in summary:
    display(Markdown(f"### {row['optimizer']}"))
    from IPython.display import Image as DisplayImage
    display(DisplayImage(filename=str(RUN_DIR / row["optimizer"] / "loss_curves.png")))
print("Full report:", RUN_DIR / "experiment_report.md")


## 6. Download small integration artifacts
This ZIP contains metrics, plots, CSVs, reports, environment and provenance. It excludes weights/checkpoints, which remain on Drive. After extraction in your local clone, place the run under `results/model_b/<RUN_ID>/`. Review and commit only those new Model B artifacts. Do not replace teammate files or split manifests. Keep `best_model.pt`, `last_model.pt` and `model_weights.pt` on Drive for future evaluation/recovery.

The final report discussion must be based on these real results. A single seed and an image-level random split do not establish geographical generalization or deployment latency.


In [ ]:
archive_path = Path("/content") / f"{RUN_ID}_integration.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and path.suffix.lower() in {".json", ".csv", ".png", ".md"}:
            archive.write(path, Path("results/model_b") / RUN_ID / path.relative_to(RUN_DIR))
print("Checkpoints remain on Drive at", RUN_DIR)
files.download(str(archive_path))
